# Containerization & Docker: Packaging ML Systems

## Learning Objectives
- Build Docker images with reproducible environments
- Optimize image size and build times
- Design multi-model containerization strategies

## Interview Questions This Covers
- "Code works on laptop, fails on server. Docker fixes it. How?"
- "Docker image is 5GB. Optimize."
- "Design containerization for 100 models"

## Basic: Dockerfile Structure & Layer Caching

In [ ]:
# Example Dockerfile for ML model training
dockerfile_content = '''
FROM python:3.9-slim

WORKDIR /app

# Copy dependencies first (slow to install, change rarely)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy code last (changes frequently)
COPY train.py .
COPY data/ data/

ENV SEED=42
ENV MODEL_PATH=/app/model.pkl

ENTRYPOINT ["python", "train.py"]
'''

print("DOCKERFILE BEST PRACTICES")
print()
print(dockerfile_content)
print()
print("Why this order?")
print("1. Base image (Python 3.9-slim): 100MB, never changes")
print("2. Dependencies: 500MB, change occasionally")
print("3. Code: 10MB, changes frequently")
print()
print("Docker caching:")
print("  - If code changes: reuse cached layers 1-2, rebuild layer 3")
print("  - If dependencies change: rebuild layers 2-3")
print()
print("Build times:")
print("  - First build: 5 minutes (install everything)")
print("  - Code change: 10 seconds (reuse cached dependencies)")
print("  - Without optimization: 5 minutes (rebuild everything)")

## Advanced: Multi-Stage Build for Size Optimization

In [ ]:
multi_stage = '''
# Stage 1: Builder (large, contains build tools)
FROM python:3.9 as builder

WORKDIR /build
COPY requirements.txt .

# Install + compile wheels
RUN pip install --user --no-cache-dir -r requirements.txt

# Stage 2: Runtime (small, only runtime dependencies)
FROM python:3.9-slim

WORKDIR /app

# Copy only the installed packages from builder
COPY --from=builder /root/.local /root/.local

# Copy application
COPY train.py .

ENTRYPOINT ["python", "train.py"]
'''

print("MULTI-STAGE BUILD")
print()
print(multi_stage)
print()
print("Size comparison:")
print("  - Full image (stage 1): 1.2GB")
print("    - Python 3.9: 900MB")
print("    - Build tools (gcc, make): 300MB")
print("    - Packages: 500MB")
print()
print("  - Multi-stage image (stage 2): 500MB")
print("    - Python 3.9-slim: 100MB")
print("    - Packages: 400MB")
print()
print("  - Reduction: 1.2GB → 500MB (58% smaller!)")
print("  - Faster push/pull to registry")

## Real-World Examples: Netflix, Stripe, Uber

In [ ]:
def netflix_containerization():
    print("NETFLIX: Containerizing 7000+ Titles")
    print("- Multi-stage builds reduce image from 2GB → 400MB")
    print("- Pin all dependency versions exactly")
    print("- Health checks: curl localhost:8000/health")
    print("- Deploy same image across 100 servers")
    print("- Results: reproducible, fast scaling")
    print()

def stripe_containerization():
    print("STRIPE: Python 3.8 → 3.9 Migration")
    print("- Old Dockerfile: FROM python:3.8")
    print("- New Dockerfile: FROM python:3.9")
    print("- Build new image, test locally")
    print("- Canary: 5% traffic on new container")
    print("- Monitor: no errors for 24h")
    print("- Full rollout: expand to 100%")
    print("- Benefit: container isolates env, safe migration")
    print()

def uber_containerization():
    print("UBER: Multi-Model Container Strategy")
    print("- Base image: FROM pytorch-serving:latest")
    print("- Models mounted at /models/model_name")
    print("- Same image deployed 50 times (different models)")
    print("- Model update: new version pushed to S3, pod restarts")
    print("- No container rebuild needed")
    print("- Reduces image count from 50 → 1 (easier management)")

netflix_containerization()
stripe_containerization()
uber_containerization()

## Interview Case Study: Containerizing ML Pipeline

In [ ]:
print("CASE STUDY: CONTAINERIZE FRAUD MODEL TRAINING")
print()
print("Dockerfile:")
print("  FROM python:3.9-slim")
print("  RUN pip install --no-cache-dir pandas scikit-learn xgboost")
print("  COPY train.py /app/")
print("  ENV SEED=42")
print("  ENTRYPOINT ['python', 'train.py']")
print()
print("Benefits:")
print("  ✓ Reproducibility: same dependencies everywhere")
print("  ✓ Portability: runs on laptop, CI/CD, Kubernetes")
print("  ✓ Version control: git commit → docker image version")
print("  ✓ Rollback: previous image available instantly")
print()
print("Workflow:")
print("  1. git commit -m 'improve fraud model'")
print("  2. docker build -t fraud:abc123 .  (tag with commit hash)")
print("  3. docker run fraud:abc123 (train locally, verify)")
print("  4. docker push myregistry/fraud:abc123")
print("  5. kubernetes deploy fraud:abc123")
print("  6. Monitor metrics")
print("  7. If issues: kubernetes deploy fraud:def456 (previous version)")

## Key Takeaways

**Docker enables reproducibility:** Same image = same behavior everywhere.

**Layer caching speeds development:** Code changes rebuild in seconds, not minutes.

**Multi-stage builds optimize size:** Remove build tools, keep only runtime dependencies.

**Common pattern:** Dockerfile with pinned versions + build locally + push to registry + deploy.